# End of week 1 exercise

A programming tutor that answers any technical question using two AI models — GPT-4o-mini (OpenAI) and Llama 3.2 (local via Ollama) — so you can compare their responses side by side.

---

### Prerequisites
- An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`
- [Ollama](https://ollama.com) installed and running locally with `llama3.2` pulled (`ollama pull llama3.2`)


## 1. Imports

Standard library and third-party dependencies:
- `os` / `dotenv` — load the API key from the `.env` file
- `openai.OpenAI` — used for both OpenAI and Ollama (Ollama exposes an OpenAI-compatible REST API)
- `IPython.display` — render the model response as formatted markdown inside the notebook

In [ ]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

## 2. Configuration & Prompt

**Model constants** pin the exact model versions so behaviour is reproducible.

**`SYSTEM_PROMPT`** defines the tutor's persona and response style. It instructs the model to:
- Target an intermediate developer audience
- Use concrete examples
- Surface edge cases and gotchas
- Format output as markdown

**`build_messages()`** is a shared helper that assembles the `messages` list expected by the OpenAI Chat Completions API:

```python
[
    {"role": "system", "content": "...tutor persona..."},
    {"role": "user",   "content": "...the question..."},
]
```

Both the GPT and Llama functions call this helper, keeping prompt logic in one place.

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

SYSTEM_PROMPT = """You are an expert programming tutor with deep knowledge of Python, \
software engineering, and computer science fundamentals.

When answering a technical question:
- Provide a clear, concise explanation suitable for an intermediate developer
- Use concrete examples to illustrate concepts
- Highlight any important edge cases or gotchas
- Respond in markdown for readability"""


def build_messages(question: str) -> list[dict]:
    """Build the messages list for the chat completion API."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question.strip()},
    ]

## 3. Environment Setup

Loads credentials and creates two API clients:

| Client | Endpoint | Purpose |
|--------|----------|---------|
| `openai_client` | `api.openai.com` | Calls GPT-4o-mini in the cloud |
| `ollama_client` | `localhost:11434/v1` | Calls Llama 3.2 running locally |

Ollama exposes the same OpenAI-compatible REST interface, so we can reuse the `openai.OpenAI` SDK for both — no extra dependencies needed.

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key found - please check your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("API key found but doesn't start with 'sk-proj-' - please verify.")
elif api_key.strip() != api_key:
    print("API key has leading/trailing whitespace - please remove it.")
else:
    print("API key found and looks good!")

openai_client = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama_client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

## 4. Ask a Question

Edit the `question` variable below with any technical programming question.  
Multi-line strings work well — you can include code snippets using triple-quoted strings as shown.

In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

## 5. Answer from GPT-4o-mini (OpenAI)

`ask_gpt()` calls the OpenAI API with **streaming enabled** (`stream=True`).  
Streaming means the response is delivered token-by-token as it's generated.  
Here we accumulate all chunks into a single string and render it as markdown at the end — giving us clean formatted output without flickering intermediate states.

In [ ]:
# Get gpt-4o-mini to answer, with streaming

def ask_gpt(question: str) -> None:
    """Stream a GPT response and display it as rendered markdown."""
    messages = build_messages(question)
    stream = openai_client.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True,
    )

    response_text = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        response_text += delta

    display(Markdown(response_text))


print(f"--- Answer from {MODEL_GPT} ---")
ask_gpt(question)

## 6. Answer from Llama 3.2 (Ollama — local)

`ask_llama()` calls the locally-running Llama 3.2 model through Ollama.  
Unlike the GPT call above, this runs **entirely on your machine** — no data leaves your environment and there is no API cost.

> **Tip:** If Ollama is not running, start it with `ollama serve` in a terminal.  
> If you haven't pulled the model yet, run `ollama pull llama3.2` first.

In [ ]:
# Get Llama 3.2 to answer

def ask_llama(question: str) -> None:
    """Call a local Llama model via Ollama and display the response as markdown."""
    messages = build_messages(question)
    response = ollama_client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
    )
    answer = response.choices[0].message.content
    display(Markdown(answer))


print(f"--- Answer from {MODEL_LLAMA} ---")
ask_llama(question)